# Notebook 5 — SHAP Explanation & Drift Management
**What this notebook does:**
- SHAP token-level explanation of drift
- Identifies which words cause drift
- Severity-based drift management strategy
- Final comparison of all models
- Complete thesis results summary

## Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('All imports loaded ✓')

## Step 2 — Load Everything

In [ ]:
# Load data
df_train = pd.read_csv('../data/train_data.csv')
df_test  = pd.read_csv('../data/test_data.csv')

# Load models
with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('../models/logistic_regression.pkl', 'rb') as f:
    lr_model = pickle.load(f)
with open('../models/random_forest.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Load drift results
drift_df = pd.read_csv('../outputs/drift_detection_results.csv')

# Transform data
X_train = tfidf.transform(df_train['text'].fillna(''))
X_test  = tfidf.transform(df_test['text'].fillna(''))
y_train = df_train['label'].values
y_test  = df_test['label'].values

feature_names = tfidf.get_feature_names_out()
print('Everything loaded ✓')
print(f'Features available: {len(feature_names):,}')

## Step 3 — SHAP Explanation on Training Data

In [ ]:
print('Computing SHAP values on training data...')
print('(This takes 2-3 minutes)')

# Sample for SHAP computation
SHAP_SAMPLES = 500
sample_idx_train = np.random.choice(X_train.shape[0], SHAP_SAMPLES, replace=False)
X_shap_train = X_train[sample_idx_train]

# Create SHAP explainer
explainer = shap.LinearExplainer(
    lr_model, X_train,
    feature_perturbation='interventional'
)

shap_values_train = explainer.shap_values(X_shap_train)
print('SHAP values computed for training data ✓')

## Step 4 — SHAP Explanation on Drifted Data

In [ ]:
print('Computing SHAP values on test/drift data...')

sample_idx_test = np.random.choice(X_test.shape[0], SHAP_SAMPLES, replace=False)
X_shap_test = X_test[sample_idx_test]

shap_values_test = explainer.shap_values(X_shap_test)
print('SHAP values computed for drift data ✓')

## Step 5 — Top SHAP Features Per Class

In [ ]:
class_names = ['Negative', 'Neutral', 'Positive']

print('TOP 10 SHAP FEATURES — TRAINING DATA (1999-2018)')
print('='*55)
for cls_idx, cls_name in enumerate(class_names):
    mean_shap = np.abs(shap_values_train[cls_idx]).mean(axis=0)
    top10 = mean_shap.argsort()[-10:][::-1]
    print(f'\n{cls_name} Class:')
    for rank, i in enumerate(top10, 1):
        print(f'  {rank:2}. {feature_names[i]:<25} SHAP: {mean_shap[i]:.4f}')

print('\n\nTOP 10 SHAP FEATURES — TEST/DRIFT DATA (2019-2023)')
print('='*55)
for cls_idx, cls_name in enumerate(class_names):
    mean_shap = np.abs(shap_values_test[cls_idx]).mean(axis=0)
    top10 = mean_shap.argsort()[-10:][::-1]
    print(f'\n{cls_name} Class:')
    for rank, i in enumerate(top10, 1):
        print(f'  {rank:2}. {feature_names[i]:<25} SHAP: {mean_shap[i]:.4f}')

## Step 6 — SHAP Drift Comparison Graph

In [ ]:
# Compare top features SHAP values: Training vs Drift
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for cls_idx, (ax, cls_name) in enumerate(zip(axes, class_names)):
    # Get mean SHAP for both periods
    train_shap = np.abs(shap_values_train[cls_idx]).mean(axis=0)
    test_shap  = np.abs(shap_values_test[cls_idx]).mean(axis=0)

    # Get top 10 features by training SHAP
    top10_idx   = train_shap.argsort()[-10:][::-1]
    top10_names = [feature_names[i] for i in top10_idx]
    top10_train = train_shap[top10_idx]
    top10_test  = test_shap[top10_idx]

    x = np.arange(len(top10_names))
    width = 0.35

    ax.barh(x + width/2, top10_train, width, label='Training (1999-2018)', color='#2E75B6')
    ax.barh(x - width/2, top10_test,  width, label='Drift (2019-2023)',    color='#C55A11')
    ax.set_yticks(x)
    ax.set_yticklabels(top10_names, fontsize=9)
    ax.set_title(f'{cls_name} Class\nSHAP Feature Shift', fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|')
    ax.legend(fontsize=8)

plt.suptitle(
    'SHAP Token-Level Drift Explanation\n'
    'Blue=Training Period | Orange=Drift Period',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../outputs/05_shap_drift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP comparison graph saved ✓')

## Step 7 — SHAP Summary Plot

In [ ]:
# SHAP summary for Positive class on drift data
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_test[2],  # Positive class
    X_shap_test,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title('SHAP Summary — Positive Class (Drift Period 2019-2023)',
          fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/05_shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP summary plot saved ✓')

## Step 8 — Drift Management Strategy

In [ ]:
WINDOW_SIZE = 5000
BASELINE_ACC = 0.86

def apply_drift_management(severity, window_num, model, X_new, y_new, X_train, y_train):
    """
    Three-tier SHAP-guided drift management:
    LOW    → Monitor only (no retraining)
    MEDIUM → Partial retrain on recent window
    HIGH   → Full retrain on all data
    """
    print(f'\nWindow {window_num} | Severity: {severity}')
    print('-'*45)

    if severity == 'LOW':
        print('ACTION: Monitor Only')
        print('→ Model performance acceptable')
        print('→ No retraining needed')
        print('→ Continue monitoring next window')
        action = 'Monitor'

    elif severity == 'MEDIUM':
        print('ACTION: Partial Retraining')
        print('→ SHAP detected moderate feature shift')
        print('→ Retraining on recent window only')
        from sklearn.linear_model import LogisticRegression
        model = LogisticRegression(max_iter=500, class_weight='balanced')
        model.fit(X_new, y_new)
        new_acc = accuracy_score(y_new, model.predict(X_new))
        print(f'→ Model updated | New accuracy: {new_acc:.2%}')
        action = 'Partial Retrain'

    elif severity == 'HIGH':
        print('ACTION: Full Retraining')
        print('→ SHAP detected major feature distribution shift')
        print('→ Retraining on all available data')
        from scipy.sparse import vstack
        from sklearn.linear_model import LogisticRegression
        X_combined = vstack([X_train, X_new])
        y_combined = np.concatenate([y_train, y_new])
        model = LogisticRegression(max_iter=500, class_weight='balanced')
        model.fit(X_combined, y_combined)
        new_acc = accuracy_score(y_new, model.predict(X_new))
        print(f'→ Model fully retrained | New accuracy: {new_acc:.2%}')
        action = 'Full Retrain'

    return model, action

print('Drift management function defined ✓')

## Step 9 — Apply Management to All Windows

In [ ]:
print('APPLYING DRIFT MANAGEMENT STRATEGY')
print('='*55)

current_model = lr_model
management_log = []

for _, row in drift_df.iterrows():
    w_num = int(row['window'][1:]) - 1
    start = w_num * WINDOW_SIZE
    end   = start + WINDOW_SIZE

    X_win = X_test[start:end]
    y_win = y_test[start:end]

    acc_before = accuracy_score(y_win, current_model.predict(X_win))

    current_model, action = apply_drift_management(
        row['severity'], row['window'],
        current_model, X_win, y_win, X_train, y_train
    )

    acc_after = accuracy_score(y_win, current_model.predict(X_win))

    management_log.append({
        'window':      row['window'],
        'year_range':  row['year_range'],
        'severity':    row['severity'],
        'action':      action,
        'acc_before':  round(acc_before, 4),
        'acc_after':   round(acc_after, 4),
        'improvement': round(acc_after - acc_before, 4)
    })

mgmt_df = pd.DataFrame(management_log)
mgmt_df.to_csv('../outputs/management_results.csv', index=False)
print('\nManagement results saved ✓')

## Step 10 — Management Results Graph

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(mgmt_df))
width = 0.35

bars1 = ax.bar(x - width/2, mgmt_df['acc_before'], width,
               label='Before Management', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, mgmt_df['acc_after'],  width,
               label='After Management',  color='#2ecc71', alpha=0.8)

ax.axhline(y=BASELINE_ACC, color='blue', linestyle='--',
            linewidth=2, label=f'Original Baseline ({BASELINE_ACC:.0%})')

ax.set_xlabel('Time Window')
ax.set_ylabel('Accuracy')
ax.set_title('Drift Management Results\nAccuracy Before vs After Management Strategy',
              fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['window']}\n{r['severity']}\n{r['action']}" for _, r in mgmt_df.iterrows()],
    fontsize=8
)
ax.legend()
ax.set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig('../outputs/05_management_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Management results graph saved ✓')

## Step 11 — Complete Thesis Results Summary

In [ ]:
print('='*65)
print('COMPLETE THESIS IMPLEMENTATION SUMMARY')
print('='*65)

print('\n📊 DATASET:')
print(f'  Source:        Amazon Electronics Reviews (Stanford SNAP)')
print(f'  Total records: 200,000 reviews')
print(f'  Year range:    1999 — 2023 (24 years)')
print(f'  Training set:  {len(df_train):,} reviews (1999-2018)')
print(f'  Test/Drift:    {len(df_test):,} reviews (2019-2023)')

print('\n🤖 BASELINE MODELS:')
print(f'  Logistic Regression:  86% accuracy | F1: 0.84')
print(f'  Random Forest:        85% accuracy | F1: 0.81')

print('\n🔍 DRIFT DETECTION:')
print(f'  Method:        KS Test + PSI + Sliding Window')
print(f'  Windows:       {len(drift_df)} windows analyzed')
print(f'  LOW severity:  {(drift_df["severity"]=="LOW").sum()} windows')
print(f'  MEDIUM:        {(drift_df["severity"]=="MEDIUM").sum()} windows')
print(f'  HIGH:          {(drift_df["severity"]=="HIGH").sum()} windows')

print('\n💡 SHAP EXPLANATION:')
print(f'  Method:        LinearSHAP on TF-IDF features')
print(f'  Samples:       {SHAP_SAMPLES} training + {SHAP_SAMPLES} drift')
print(f'  Output:        Token-level feature attribution shift')

print('\n⚙️ DRIFT MANAGEMENT:')
for _, r in mgmt_df.iterrows():
    improvement = r['improvement']
    sign = '+' if improvement >= 0 else ''
    print(f'  {r["window"]} ({r["severity"]:<6}): {r["action"]:<15} '
          f'Acc: {r["acc_before"]:.2%} → {r["acc_after"]:.2%} '
          f'({sign}{improvement*100:.1f}%)')

print('\n📁 OUTPUT FILES SAVED:')
import os
for f in sorted(os.listdir('../outputs')):
    print(f'  ✓ outputs/{f}')

print('\n' + '='*65)
print('✅ ALL NOTEBOOKS COMPLETE!')
print('✅ Thesis 1 Implementation Done!')
print('='*65)